# 17. Chain of Thought

**Tier:** Building with LLMs
**Estimated time:** 40 minutes
**Prerequisites:** 06, 12
**Source material:** @sairahul1 — https://x.com/sairahul1/status/2057740928908161461 ; Stanford LLM curriculum, Lecture 6 (LLM reasoning)

## What You'll Learn
- Why asking a model to "show its work" measurably improves multi-step answers
- The difference between zero-shot CoT, few-shot CoT, and self-consistency
- Where reasoning models fit, and how this connects to the verification loops in notebook 23

## Why This Matters
A model generates one token at a time (notebook 06), committing to each word before seeing the next. For a multi-step problem, forcing an answer immediately means it has to "decide" the conclusion before it has reasoned through the steps. Chain of Thought gives it room to compute the intermediate steps in the open — the single cheapest accuracy boost available on hard problems.


## Thinking out loud, because the model can't think silently

Here's the key intuition from notebook 06: an LLM has no scratchpad. It produces the answer token by token, and each token is conditioned only on the tokens *already written*. So if you demand "What's the answer?" for a problem that needs five reasoning steps, the model must emit the answer token before any of those steps exist in its context — it's guessing the conclusion, then maybe rationalizing.

**Chain of Thought (CoT)** fixes this by letting the model write the steps *first*. The reasoning tokens become part of the context the final-answer token is conditioned on — the model literally has more relevant computation in front of it when it commits. Two flavors:

- **Zero-shot CoT**: just append "Let's think step by step." That one phrase triggers the model to lay out intermediate steps before answering.
- **Few-shot CoT**: show worked examples that include the reasoning, not just the final answer (notebook 12's few-shot pattern, applied to reasoning).

**Self-consistency** takes it further: sample several independent chains of thought (with a non-zero temperature so they differ), then take a majority vote on the final answers. Different reasoning paths that converge on the same answer are more trustworthy than a single chain — wrong chains tend to disagree with each other, right chains tend to agree.

Modern **reasoning models** bake this in: they're trained (notebook 09's RL ideas, extended) to produce long internal reasoning before answering, so you often don't need to prompt for it explicitly.


In [ ]:
import os, re, collections
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # production default: claude-opus-4-8

def ask(prompt, system="You are a helpful assistant.", max_tokens=400, temperature=1.0, model=TEACH_MODEL):
    if not HAS_ANTHROPIC:
        print("  [skipped: no ANTHROPIC_API_KEY]")
        return None
    try:
        import anthropic
        client = anthropic.Anthropic()
        msg = client.messages.create(model=model, max_tokens=max_tokens, system=system,
                                      temperature=temperature,
                                      messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        print(f"  [skipped: API call failed — {type(e).__name__}: {str(e)[:150]}]")
        return None


## Minimal example: direct answer vs. step-by-step

A word problem with a couple of dependent steps — the kind where committing to an answer immediately is risky.


In [ ]:
PROBLEM = (
    "A workshop has 3 robots. Each robot assembles 4 units per hour and runs for 6 hours, "
    "but one robot is down for 2 of those hours. How many units are assembled in total?"
)

DIRECT_PROMPT = PROBLEM + " Answer with just the final number."
COT_PROMPT = PROBLEM + " Let's think step by step, then give the final number on its own line."

print("--- Direct (answer immediately) ---")
print(ask(DIRECT_PROMPT, max_tokens=50))
print("\n--- Chain of Thought (reason first) ---")
print(ask(COT_PROMPT))


## Self-consistency: sample several chains, take the majority

We sample the CoT prompt a few times at temperature 1.0 so the reasoning paths differ, extract each final number, and vote. (The correct answer here is 2×4×6 + 1×4×4 = 48 + 16 = 64.)


In [ ]:
def extract_final_number(text):
    if not text:
        return None
    nums = re.findall(r"-?\d+", text.replace(",", ""))
    return int(nums[-1]) if nums else None

def self_consistency(prompt, n_samples=5):
    answers = []
    for _ in range(n_samples):
        out = ask(prompt, temperature=1.0)
        ans = extract_final_number(out)
        if ans is not None:
            answers.append(ans)
    if not answers:
        print("  [no samples — API unavailable]")
        return None, answers
    vote = collections.Counter(answers).most_common(1)[0][0]
    return vote, answers

majority, raw_answers = self_consistency(COT_PROMPT, n_samples=5)
print("Individual chain answers:", raw_answers)
print("Majority-vote answer:", majority, "(expected: 64)")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import collections as _c

if raw_answers:
    counts = _c.Counter(raw_answers)
    plt.figure(figsize=(6, 4))
    plt.bar([str(k) for k in counts], list(counts.values()), color="#4C72B0")
    plt.title("Self-consistency: distribution of answers across reasoning chains")
    plt.xlabel("Final answer"); plt.ylabel("Number of chains")
    plt.tight_layout(); plt.show()
else:
    print("No answers to plot (API unavailable).")


*When most independent chains converge on one answer, that agreement is a stronger signal than any single chain — the minority answers are usually the ones that took a wrong reasoning turn.*


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Try a harder problem
# Task: Replace PROBLEM with a 3-step problem of your own (e.g. involving a discount AND tax).
#       Compare the direct vs. CoT answers. Does the gap widen on the harder problem?
# Hint: CoT helps most when the number of dependent steps is high; on a 1-step problem it
#       barely matters.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Few-shot CoT
# Task: Write a prompt with ONE fully worked example (problem + step-by-step solution + answer),
#       then pose a new problem. Does showing the reasoning style improve the format/accuracy of
#       the new answer vs. zero-shot "let's think step by step"?
# Hint: This is notebook 12's few-shot pattern applied to the *reasoning*, not just the answer.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): Self-consistency vs. cost
# Task: Run self_consistency with n_samples = 1, 3, and 7 on the same problem. Plot how often the
#       majority answer is correct against the number of samples. Where do diminishing returns hit?
# Hint: More samples = more robust vote but linearly more cost/latency. This same "sample many,
#       pick the best" idea reappears as the Tournament and Generate-and-Filter loops in notebook 23.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
PROBLEM2 = ("A jacket costs $80. There's a 25% discount, then 8% sales tax is added to the "
            "discounted price. What is the final price?")
print(ask(PROBLEM2 + " Just the number.", max_tokens=30))
print(ask(PROBLEM2 + " Let's think step by step, final number on its own line."))
# Correct: 80*0.75 = 60; 60*1.08 = 64.80. CoT is far more reliable on the 2-step version.

# Exercise 2
FEWSHOT = '''Example:
Problem: 2 machines make 3 parts/hr for 5 hrs. How many parts?
Solution: Each machine: 3*5 = 15. Two machines: 15*2 = 30.
Answer: 30

Now solve:
Problem: 4 machines make 5 parts/hr for 3 hrs, but one machine is idle the whole time. How many parts?
Solution:'''
print(ask(FEWSHOT))   # the worked example pins down the reasoning format

# Exercise 3
import matplotlib.pyplot as plt
correct = 64
rates = []
for n in [1, 3, 7]:
    votes = [self_consistency(COT_PROMPT, n_samples=n)[0] for _ in range(3)]
    rates.append(sum(v == correct for v in votes) / len(votes))
plt.plot([1,3,7], rates, marker="o"); plt.xlabel("n_samples"); plt.ylabel("fraction correct")
plt.title("Self-consistency accuracy vs sample count"); plt.show()
# Accuracy usually rises then plateaus — past a point, extra samples mostly add cost.
```
</details>


## Key Takeaways
- A model has no hidden scratchpad — it conditions each token only on what's already written, so making it write the reasoning first gives the final answer more to stand on.
- Zero-shot CoT ("let's think step by step") is nearly free and helps most as the number of dependent steps grows.
- Few-shot CoT pins down a reasoning *style*; self-consistency samples multiple chains and votes, trading cost for robustness.
- Reasoning models internalize CoT through training, so you often get step-by-step reasoning without prompting for it.
- "Sample many, then pick" reappears as a core agent pattern — the Tournament and Generate-and-Filter loops in notebook 23.

## What's Next
That closes Tier 3 — you can now prompt, cache, retrieve, and reason. Notebook **18 — What Is an Agent** opens Tier 4 by giving the model something new: the ability to *act* — to call tools, observe results, and decide what to do next, in a loop.
